In [ ]:
#08/03/2025

In [ ]:
import numpy as np
import os

from sian.synth import HCAM_MultilinearSEM_ERgraphSCM_InputDataGenerator
from sian.synth import HCAM_DiscreteSEM_ERgraphSCM_InputDataGenerator
from sian.synth import save_synthetic_as_official_dataset

%load_ext autoreload
%autoreload 2

### classification (discrete variables)

In [ ]:
SAVING = False
SAVING = True

cls_base_path  = "hoc_data/classification_N_10k/"
if not os.path.exists(cls_base_path):
    os.makedirs(cls_base_path)

In [ ]:
N=10*1000

all_cls_configs=[]
for D in [10,30]:
    for I_k in [5]:
        # for ER_degree in [1,4]:
        for ER_degree in [1,2,4]:
            for my_seed in range(5):
                for GAM_degree in [1,2,3]:
                    config_tuple = N,D,I_k,ER_degree,GAM_degree,my_seed
                    all_cls_configs.append(config_tuple)
print("ALL CONFIGS")
print(all_cls_configs)

In [ ]:

for config_tuple in all_cls_configs:
    N,D,I_k,ER_degree,GAM_degree,my_seed = config_tuple
    exp_name = "exp_" + "cls_"+ f"D{D}_" + f"ER{ER_degree}_" + f"easyGAM{GAM_degree}_" + f"seed{my_seed}"
    file_path = cls_base_path + exp_name + "/"

    dgp_x = HCAM_DiscreteSEM_ERgraphSCM_InputDataGenerator(D, I_k, GAM_degree, ER_degree, metainit_seed=my_seed)
    X30_dict = dgp_x.generate_samples(N)
    X30 = np.concatenate(list(X30_dict.values()),axis=-1)
    
    
    if SAVING:
        if not os.path.exists(file_path):
            os.mkdir(file_path)
        
        dgp_x.save_details(file_path)
        np.save(file_path+'X.npy',X30)
        
        save_synthetic_as_official_dataset(file_path, exp_name, X30, dgp_x)
    

### regression (continuous variables)

In [ ]:
SAVING = False
SAVING = True

reg_base_path  = "hoc_data/regression_N_10k/"
if not os.path.exists(reg_base_path):
    os.makedirs(reg_base_path)

In [ ]:
N = 10*1000

all_reg_configs=[]
for D in [10,30]:
    for ER_degree in [1,4]:
        for my_seed in range(5):
            for GAM_degree in [2,3]:
                config_tuple = N,D,ER_degree,GAM_degree,my_seed
                all_reg_configs.append(config_tuple)
print("ALL CONFIGS")
print(all_reg_configs)

In [ ]:

for config_tuple in all_reg_configs:
    N,D,ER_degree,GAM_degree,my_seed=config_tuple
    exp_name = "exp_" + "reg_"+ f"D{D}_" + f"ER{ER_degree}_" + f"easyGAM{GAM_degree}_" + f"seed{my_seed}"
    file_path = reg_base_path + exp_name + "/"

    dgp_x = HCAM_MultilinearSEM_ERgraphSCM_InputDataGenerator(D, GAM_degree, ER_degree, metainit_seed=my_seed)
    X30, N30 = dgp_x.generate_samples(N, return_errors=True)

    if SAVING:
        if not os.path.exists(file_path):
            os.mkdir(file_path)
            
        dgp_x.save_details(file_path)
        np.save(file_path+'N.npy',N30)
        np.save(file_path+'X.npy',X30)
        
        save_synthetic_as_official_dataset(file_path, exp_name, X30, dgp_x)
